In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pprint import pp
from uuid import UUID

from evo.data_converters.common import create_evo_object_service_and_data_client
from evo.notebooks import ServiceManagerWidget

client_id = os.getenv("EVO_CLIENT_ID", "")
base_uri = os.getenv("EVO_BASE_URI", "")
discovery_url = os.getenv("EVO_DISCOVERY_URL", "")

manager = await ServiceManagerWidget.with_auth_code(
    client_id=client_id, base_uri=base_uri, discovery_url=discovery_url
).login()

object_service_client, data_client = create_evo_object_service_and_data_client(service_manager_widget=manager)

In [ ]:
available_objects = await object_service_client.list_objects(schema_id=["like:*downhole-collection*"])

# picked_object = available_objects[1]
# 0aa73ada-aa76-46f6-a985-5db941b80ed4 has multiple holes
# 060a3bf0-0d6f-4874-8112-320e8db97064 has voids
# 572b1fb5-aa52-4a2f-9eca-9e744834cf1b has interval tables
picked_object = next(iter([o for o in available_objects if o.id == UUID("64e8a846-bb0c-4478-b8c0-f8ceec8c6b56")]))

print(f"Picked downhole-collection {picked_object.id} of {available_objects.size} available objects.")

downloaded_object = await object_service_client.download_object_by_id(
    object_id=picked_object.id, version=picked_object.version_id
)
pp(downloaded_object)

In [ ]:
from evo.data_converters.common.objects.downhole_collection_from_evo import create_downhole_collection_from_evo

dhc_trial = await create_downhole_collection_from_evo(downloaded_object)
pp(dhc_trial.name)
pp(dhc_trial.uuid)
pp(dhc_trial.collars.df)
for table in dhc_trial.measurements:
    pp(type(table))
    pp(table.df)